# Lista 7 — Modelo CER Multivariado Correlacionado
**ICP 103: Análise de Risco — Prof. Eber**

## Exercício 1: Ações da B3 (PETR3, VALE3, ABEV3, BBAS3)

# 0. Pacotes Necessários

In [ ]:
install.packages(c("quantmod", "MASS", "ggplot2", "reshape2"), quiet = TRUE)
library(quantmod)
library(MASS)       # mvrnorm — amostragem multivariada
library(ggplot2)
library(reshape2)

# 1. Download dos dados históricos (últimos 12 meses)

In [ ]:
# --- data fixa para reprodutibilidade ---
data_fim    <- as.Date("2026-06-16")   # semana passada; troque conforme necessário
data_inicio <- data_fim - 365

tickers <- c("PETR3.SA", "VALE3.SA", "ABEV3.SA", "BBAS3.SA")
nomes   <- c("Petrobras", "Vale", "Ambev", "Banco do Brasil")

precos_raw <- list()
for (t in tickers) {
  precos_raw[[t]] <- getSymbols(t, src = "yahoo",
                                from = data_inicio, to = data_fim + 1,
                                auto.assign = FALSE)
}

# Preços de fechamento ajustados, alinhados numa única série xts
fechamentos <- do.call(merge, lapply(precos_raw, Ad))
colnames(fechamentos) <- nomes

cat("Período:", as.character(index(fechamentos)[1]),
    "a", as.character(index(fechamentos)[nrow(fechamentos)]), "\n")
cat("Observações:", nrow(fechamentos), "dias úteis\n")
tail(fechamentos)

# 2. Estimação dos parâmetros do modelo CER

In [ ]:
# Retornos continuamente compostos (log-retornos)
# r_t = ln(P_t) - ln(P_{t-1})
retornos <- diff(log(fechamentos))
retornos <- na.omit(retornos)

# Parâmetros CER por ativo
mu_vec    <- colMeans(retornos)          # média diária
sigma_vec <- apply(retornos, 2, sd)      # desvio padrão diário

params <- data.frame(
  Ativo        = nomes,
  mu_diario    = round(mu_vec, 6),
  sigma_diario = round(sigma_vec, 6),
  mu_anual     = round(mu_vec * 252, 4),
  sigma_anual  = round(sigma_vec * sqrt(252), 4)
)
rownames(params) <- NULL
print(params)

# 3. Teste de Normalidade dos Retornos

In [ ]:
# Shapiro-Wilk por ativo
# H0: a amostra segue distribuição normal
# p-valor > 0.05 → não rejeita normalidade
cat("=== Shapiro-Wilk (H0: normalidade) ===\n")
for (nm in nomes) {
  sw <- shapiro.test(as.numeric(retornos[, nm]))
  cat(sprintf("%-18s W = %.4f  p-valor = %.4f  %s\n",
              nm, sw$statistic, sw$p.value,
              ifelse(sw$p.value > 0.05, "[não rejeita H0]", "[REJEITA H0]")))
}

# Gráficos Q-Q para todos os ativos
par(mfrow = c(2, 2))
for (nm in nomes) {
  qqnorm(as.numeric(retornos[, nm]), main = paste("Q-Q:", nm),
         pch = 16, cex = 0.6, col = "steelblue")
  qqline(as.numeric(retornos[, nm]), col = "red", lwd = 2)
}
par(mfrow = c(1, 1))

# 4. Matriz de Correlação Empírica

In [ ]:
correlacao <- cor(retornos)
print(round(correlacao, 4))

# Heatmap da correlação
cor_df <- melt(correlacao)
ggplot(cor_df, aes(Var1, Var2, fill = value)) +
  geom_tile(color = "white") +
  geom_text(aes(label = round(value, 2)), size = 4) +
  scale_fill_gradient2(low = "#d73027", mid = "white", high = "#1a9850",
                       midpoint = 0, limits = c(-1, 1), name = "Correlação") +
  theme_minimal() +
  labs(title = "Matriz de Correlação Empírica — Retornos CC",
       x = NULL, y = NULL) +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))

# 5. Simulação Monte Carlo — CER Correlacionado

In [ ]:
set.seed(42)

# --- parâmetros da simulação ---
nSim      <- 1000                         # número de cenários
data_alvo <- as.Date("2026-12-31")        # horizonte: Dez/2026
n_ativos  <- length(nomes)

# dias úteis restantes (aproximação: 252 dias úteis/ano)
# contagem simples: dias corridos * (252/365)
dias_corridos <- as.numeric(data_alvo - data_fim)
T_sim <- round(dias_corridos * 252 / 365)
cat("Horizonte de simulação:", T_sim, "dias úteis (~", data_alvo, ")\n")

# P0 = último preço observado de cada ativo
P0 <- as.numeric(tail(fechamentos, 1))
names(P0) <- nomes
cat("Preços iniciais (P0):\n")
print(P0)

# Matriz de covariâncias (necessária para mvrnorm)
# Cov(i,j) = Corr(i,j) * sigma_i * sigma_j
Sigma <- cov(retornos)

# --- estrutura de armazenamento ---
# Lista de nSim cenários; cada cenário é uma matriz (T_sim x n_ativos) de PREÇOS
# Para eficiência, guardamos apenas o último dia de cada cenário
# (e uma amostra de trajetórias para o gráfico)

# Último preço de cada cenário: matriz (nSim x n_ativos)
preco_final <- matrix(NA, nrow = nSim, ncol = n_ativos)
colnames(preco_final) <- nomes

# Guardar 50 trajetórias completas para plot (uma por ativo)
n_traj_plot <- 50
traj_plot   <- array(NA, dim = c(n_traj_plot, T_sim + 1, n_ativos))

# --- loop principal ---
for (i in 1:nSim) {

  # Sorteia T_sim vetores de retornos correlacionados (T_sim x n_ativos)
  # Cada linha = 1 dia = 1 "passo do marinheiro" para os 4 ativos
  ret_sim <- mvrnorm(n = T_sim, mu = mu_vec, Sigma = Sigma)

  # Reconstrói os preços: P_t = P0 * exp(cumsum dos retornos)
  log_P0  <- log(P0)
  log_Pt  <- sweep(apply(ret_sim, 2, cumsum), 2, log_P0, "+")
  precos_cenario <- exp(log_Pt)   # (T_sim x n_ativos)

  # Salva preço final
  preco_final[i, ] <- precos_cenario[T_sim, ]

  # Salva trajetória completa para os primeiros n_traj_plot cenários
  if (i <= n_traj_plot) {
    traj_plot[i, 1, ]          <- P0                     # dia 0
    traj_plot[i, 2:(T_sim+1), ] <- precos_cenario
  }
}

cat("\nSimulação concluída:", nSim, "cenários x", T_sim, "dias\n")

# 6. Comparação: Simulação MC vs. Dados Históricos

In [ ]:
# Para cada ativo, plota:
#   - trajetórias simuladas (cinza)
#   - série histórica real (azul)

par(mfrow = c(2, 2))

for (k in 1:n_ativos) {
  nm <- nomes[k]

  # eixo x dos simulados: dias 0..T_sim
  x_sim <- 0:T_sim

  # escala y: range das trajetórias + histórico
  hist_preco <- as.numeric(fechamentos[, nm])
  ylim_range <- range(c(traj_plot[, , k], hist_preco), na.rm = TRUE)

  # trajetórias simuladas
  plot(x_sim, traj_plot[1, , k],
       type = "l", col = rgb(0.5, 0.5, 0.5, 0.3),
       ylim = ylim_range,
       xlab = "Dias a partir de P0",
       ylab = "Preço (R$)",
       main = nm)

  for (i in 2:n_traj_plot) {
    lines(x_sim, traj_plot[i, , k],
          col = rgb(0.5, 0.5, 0.5, 0.3))
  }

  # histórico real (alinhado no eixo: dias 1..n_hist relativo a data_inicio)
  n_hist <- length(hist_preco)
  lines(seq_len(n_hist), hist_preco,
        col = "steelblue", lwd = 2)

  # mediana dos cenários por dia
  mediana_sim <- apply(traj_plot[, , k], 2, median, na.rm = TRUE)
  lines(x_sim, mediana_sim, col = "red", lwd = 2, lty = 2)

  legend("topleft",
         legend = c("Histórico", "Mediana MC", "Cenários"),
         col    = c("steelblue", "red", "gray"),
         lwd    = c(2, 2, 1), lty = c(1, 2, 1),
         cex    = 0.7, bty = "n")
}

par(mfrow = c(1, 1))

# 7. Estimativa dos Preços em Dezembro/2026

In [ ]:
# Estatísticas descritivas da distribuição de preços finais
cat("=== Estimativa de preços em", as.character(data_alvo), "===", "\n\n")

for (k in 1:n_ativos) {
  nm  <- nomes[k]
  pf  <- preco_final[, k]
  cat(sprintf("%-18s  P0 = R$%6.2f  |  Média = R$%6.2f  Mediana = R$%6.2f  IC90%% [R$%6.2f, R$%6.2f]\n",
              nm, P0[k],
              mean(pf), median(pf),
              quantile(pf, 0.05), quantile(pf, 0.95)))
}

# Histogramas das distribuições de preço final
par(mfrow = c(2, 2))
for (k in 1:n_ativos) {
  nm <- nomes[k]
  hist(preco_final[, k], freq = FALSE, breaks = 40,
       col = "lightsteelblue", border = "white",
       main = paste(nm, "— Dez/2026"),
       xlab = "Preço (R$)")
  abline(v = P0[k],              col = "steelblue", lwd = 2, lty = 2)
  abline(v = mean(preco_final[, k]), col = "red",   lwd = 2)
  abline(v = quantile(preco_final[, k], c(0.05, 0.95)),
         col = "darkgreen", lwd = 1.5, lty = 3)
  legend("topright",
         legend = c("P0", "Média", "IC 5%-95%"),
         col    = c("steelblue", "red", "darkgreen"),
         lwd    = 2, lty    = c(2, 1, 3),
         cex    = 0.7, bty = "n")
}
par(mfrow = c(1, 1))

---
# Exercício 2: Criptomoedas (XRP, ETH, SOL, USDC)

# 1. Download dos dados históricos — Criptos

In [ ]:
# tickers no Yahoo Finance para criptos (cotadas em USD)
tickers_cripto <- c("XRP-USD", "ETH-USD", "SOL-USD", "USDC-USD")
nomes_cripto   <- c("XRP", "ETH", "SOL", "USDC")

# criptos negociam 7 dias/semana; usar mesmas datas
precos_raw_c <- list()
for (t in tickers_cripto) {
  precos_raw_c[[t]] <- getSymbols(t, src = "yahoo",
                                  from = data_inicio, to = data_fim + 1,
                                  auto.assign = FALSE)
}

fechamentos_c <- do.call(merge, lapply(precos_raw_c, Ad))
colnames(fechamentos_c) <- nomes_cripto

cat("Período:", as.character(index(fechamentos_c)[1]),
    "a", as.character(index(fechamentos_c)[nrow(fechamentos_c)]), "\n")
cat("Observações:", nrow(fechamentos_c), "dias (incluindo fins de semana)\n")
tail(fechamentos_c)

# 2. Parâmetros CER — Criptos

In [ ]:
retornos_c <- diff(log(fechamentos_c))
retornos_c <- na.omit(retornos_c)

mu_vec_c    <- colMeans(retornos_c)
sigma_vec_c <- apply(retornos_c, 2, sd)

# criptos: 365 dias/ano
params_c <- data.frame(
  Ativo        = nomes_cripto,
  mu_diario    = round(mu_vec_c, 6),
  sigma_diario = round(sigma_vec_c, 6),
  mu_anual     = round(mu_vec_c * 365, 4),
  sigma_anual  = round(sigma_vec_c * sqrt(365), 4)
)
rownames(params_c) <- NULL
print(params_c)

# 3. Teste de Normalidade — Criptos

In [ ]:
cat("=== Shapiro-Wilk — Criptomoedas ===\n")
for (nm in nomes_cripto) {
  sw <- shapiro.test(as.numeric(retornos_c[, nm]))
  cat(sprintf("%-6s W = %.4f  p-valor = %.4f  %s\n",
              nm, sw$statistic, sw$p.value,
              ifelse(sw$p.value > 0.05, "[não rejeita H0]", "[REJEITA H0]")))
}

par(mfrow = c(2, 2))
for (nm in nomes_cripto) {
  qqnorm(as.numeric(retornos_c[, nm]), main = paste("Q-Q:", nm),
         pch = 16, cex = 0.6, col = "darkorange")
  qqline(as.numeric(retornos_c[, nm]), col = "red", lwd = 2)
}
par(mfrow = c(1, 1))

# 4. Matriz de Correlação — Criptos

In [ ]:
correlacao_c <- cor(retornos_c)
print(round(correlacao_c, 4))

cor_df_c <- melt(correlacao_c)
ggplot(cor_df_c, aes(Var1, Var2, fill = value)) +
  geom_tile(color = "white") +
  geom_text(aes(label = round(value, 2)), size = 4) +
  scale_fill_gradient2(low = "#d73027", mid = "white", high = "#1a9850",
                       midpoint = 0, limits = c(-1, 1), name = "Correlação") +
  theme_minimal() +
  labs(title = "Matriz de Correlação Empírica — Criptomoedas",
       x = NULL, y = NULL) +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))

# 5. Simulação MC — Criptos

In [ ]:
set.seed(42)

n_ativos_c <- length(nomes_cripto)

# criptos: dias corridos até Dez/2026 (sem desconto de fins de semana)
T_sim_c <- as.numeric(data_alvo - data_fim)
cat("Horizonte de simulação (criptos):", T_sim_c, "dias corridos\n")

P0_c <- as.numeric(tail(fechamentos_c, 1))
names(P0_c) <- nomes_cripto
cat("Preços iniciais (P0 em USD):\n")
print(P0_c)

Sigma_c <- cov(retornos_c)

preco_final_c <- matrix(NA, nrow = nSim, ncol = n_ativos_c)
colnames(preco_final_c) <- nomes_cripto

traj_plot_c <- array(NA, dim = c(n_traj_plot, T_sim_c + 1, n_ativos_c))

for (i in 1:nSim) {
  ret_sim_c <- mvrnorm(n = T_sim_c, mu = mu_vec_c, Sigma = Sigma_c)

  log_P0_c  <- log(P0_c)
  log_Pt_c  <- sweep(apply(ret_sim_c, 2, cumsum), 2, log_P0_c, "+")
  precos_c  <- exp(log_Pt_c)

  preco_final_c[i, ] <- precos_c[T_sim_c, ]

  if (i <= n_traj_plot) {
    traj_plot_c[i, 1, ]             <- P0_c
    traj_plot_c[i, 2:(T_sim_c+1), ] <- precos_c
  }
}

cat("\nSimulação concluída:", nSim, "cenários x", T_sim_c, "dias\n")

# 6. Comparação: Simulação MC vs. Histórico — Criptos

In [ ]:
par(mfrow = c(2, 2))

for (k in 1:n_ativos_c) {
  nm <- nomes_cripto[k]

  # USDC é estável (~$1); escala diferente dos demais
  x_sim_c    <- 0:T_sim_c
  hist_preco <- as.numeric(fechamentos_c[, nm])
  ylim_range <- range(c(traj_plot_c[, , k], hist_preco), na.rm = TRUE)

  plot(x_sim_c, traj_plot_c[1, , k],
       type = "l", col = rgb(0.7, 0.4, 0, 0.3),
       ylim = ylim_range,
       xlab = "Dias a partir de P0",
       ylab = "Preço (USD)",
       main = nm)

  for (i in 2:n_traj_plot) {
    lines(x_sim_c, traj_plot_c[i, , k],
          col = rgb(0.7, 0.4, 0, 0.3))
  }

  n_hist_c <- length(hist_preco)
  lines(seq_len(n_hist_c), hist_preco,
        col = "steelblue", lwd = 2)

  mediana_c <- apply(traj_plot_c[, , k], 2, median, na.rm = TRUE)
  lines(x_sim_c, mediana_c, col = "red", lwd = 2, lty = 2)

  legend("topleft",
         legend = c("Histórico", "Mediana MC", "Cenários"),
         col    = c("steelblue", "red", "sienna"),
         lwd    = c(2, 2, 1), lty = c(1, 2, 1),
         cex    = 0.7, bty = "n")
}

par(mfrow = c(1, 1))

# 7. Estimativa dos Preços em Dezembro/2026 — Criptos

In [ ]:
cat("=== Estimativa de preços em", as.character(data_alvo), "(USD) ===\n\n")

for (k in 1:n_ativos_c) {
  nm  <- nomes_cripto[k]
  pf  <- preco_final_c[, k]
  cat(sprintf("%-6s P0 = $%8.4f  |  Média = $%8.4f  Mediana = $%8.4f  IC90%% [$%8.4f, $%8.4f]\n",
              nm, P0_c[k],
              mean(pf), median(pf),
              quantile(pf, 0.05), quantile(pf, 0.95)))
}

par(mfrow = c(2, 2))
for (k in 1:n_ativos_c) {
  nm <- nomes_cripto[k]
  hist(preco_final_c[, k], freq = FALSE, breaks = 40,
       col = "peachpuff", border = "white",
       main = paste(nm, "— Dez/2026"),
       xlab = "Preço (USD)")
  abline(v = P0_c[k],                   col = "steelblue", lwd = 2, lty = 2)
  abline(v = mean(preco_final_c[, k]),   col = "red",       lwd = 2)
  abline(v = quantile(preco_final_c[, k], c(0.05, 0.95)),
         col = "darkgreen", lwd = 1.5, lty = 3)
  legend("topright",
         legend = c("P0", "Média", "IC 5%-95%"),
         col    = c("steelblue", "red", "darkgreen"),
         lwd    = 2, lty    = c(2, 1, 3),
         cex    = 0.7, bty = "n")
}
par(mfrow = c(1, 1))